In [1]:
using Pkg
Pkg.activate("./../julia-code")

  Activating project at `~/code/compositional/julia-code`


In [2]:
using Meris

[ Info: Precompiling Meris [050ad861-3423-47d2-b9af-0fa14b653239] (cache misses: include_dependency fsize change (2))


In [3]:
using DataFrames, StatsBase, Distributions, LsqFit
using CairoMakie, MakiePublication, LaTeXStrings

In [4]:
# import dataset
arXive_df =  Meris.arXivSampler.collect_arXive();
# bci_df = Meris.BCITreeSampler.load_treedata(; joinquadrats=true, steps=1);
# lego_df = Meris.LegoSampler.parse_themes(; returnthemes=false);

In [5]:
domain = unique(arXive_df.domain)[1]
df = arXive_df[arXive_df.domain .== domain, :]

topic = unique(arXive_df.topic)[1]
df = df[df.topic .== topic, :]
first(df, 5)

Row,domain,topic,component_id,sample_id,counts,nreads
,String,String,String,Int64,Int64,Int64
1,physics,soc-ph,bodrum,1,1,2586
2,physics,soc-ph,whose,1,1,2586
3,physics,soc-ph,favor,1,2,2586
4,physics,soc-ph,sp,1,4,2586
5,physics,soc-ph,bronzezeit,1,1,2586


In [6]:
samples = unique(df.sample_id)
sample = samples[10]
freqs = df[df.sample_id .== sample, :].counts;

In [9]:
# Compute empirical distribution to infer parameters
ctrs, pdf = Meris.DataTools.make_hist(counts; nbins=20)

# Fit a straight line in log-space
model(x, p) = p[1] * p[2] ^ p[1] .* x .^ -(p[1] + 1)
p0 = [1.0, 1.0]
fit_model = curve_fit(model, ctrs, log.(pdf), p0)

# Compute power-law params
α, x_c = coef(fit_model)
println("α: $α     x_c: $x_c")

LoadError: DomainError with -23.578538849825:
Exponentiation yielding a complex result requires a complex argument.
Replace x^y with (x+0im)^y, Complex(x)^y, or similar.

In [8]:
# Perform test statistic
G(x) = cdf(Pareto(α, x_c), x)
Q(u) = quantile(Pareto(α, x_c), u)
pv = Meris.GOF.estimatep(collect(counts), G, Q, Meris.GOF.AndersonDarling)
println("GOF p = $pv")

LoadError: InterruptException: